In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt


META_PATH = "cnn_data/metadata.csv"
IMG_SIZE = (128, 128)   # match how you saved images
SEED = 42


def load_metadata(meta_path: str) -> pd.DataFrame:
    df = pd.read_csv(meta_path)
    required = {"writer_id", "image_path"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    # Keep only rows whose image exists
    df = df[df["image_path"].apply(lambda p: os.path.exists(p))].copy()
    if len(df) == 0:
        raise ValueError("No valid image paths found. Check your metadata.csv paths.")
    return df


def stratified_split(df: pd.DataFrame, label_col="writer_id", seed=SEED):
    """
    70/15/15 stratified split:
      - first split off test: 15%
      - then split remaining into train/val: val is 15/85 of remaining = 17.647% of remaining
        so overall ~70/15/15
    """
    y = df[label_col].values

    df_trainval, df_test = train_test_split(
        df, test_size=0.15, random_state=seed, stratify=y
    )

    y_trainval = df_trainval[label_col].values
    df_train, df_val = train_test_split(
        df_trainval, test_size=(0.15 / 0.85), random_state=seed, stratify=y_trainval
    )

    return df_train.reset_index(drop=True), df_val.reset_index(drop=True), df_test.reset_index(drop=True)


def make_tf_dataset(paths, labels, img_size=IMG_SIZE, batch_size=64, training=False):
    """
    Loads grayscale PNGs -> float32 in [0,1], shape (H,W,1)
    """
    paths = np.array(paths, dtype=str)
    labels = np.array(labels, dtype=np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load(path, label):
        img_bytes = tf.io.read_file(path)
        img = tf.io.decode_png(img_bytes, channels=1)  # grayscale
        img = tf.image.resize(img, img_size, antialias=True)
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=min(len(paths), 5000), seed=SEED, reshuffle_each_iteration=True)

        # Optional light augmentation (safe for handwriting)
        aug = keras.Sequential([
            layers.RandomTranslation(0.05, 0.05),
            layers.RandomRotation(0.05),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def build_2layer_cnn(num_classes: int, lr=1e-3, img_size=IMG_SIZE):
    inputs = keras.Input(shape=(img_size[0], img_size[1], 1))

    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Flatten()(x)

    # keep it simple; you can remove this Dense if you want literally only conv layers + softmax
    x = layers.Dense(128, activation="relu")(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def evaluate_model(model, ds, y_true):
    probs = model.predict(ds, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, cm, y_pred


def plot_confusion_matrix(cm, class_names, max_classes=40):
    """
    Confusion matrices can be huge (many writers).
    This plots the whole thing if <= max_classes else plots only the top-left max_classes.
    """
    if cm.shape[0] > max_classes:
        cm_plot = cm[:max_classes, :max_classes]
        names_plot = class_names[:max_classes]
        title = f"Confusion Matrix (first {max_classes} classes shown)"
    else:
        cm_plot = cm
        names_plot = class_names
        title = "Confusion Matrix"

    plt.figure(figsize=(10, 8))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(names_plot))
    plt.xticks(ticks, names_plot, rotation=90)
    plt.yticks(ticks, names_plot)
    plt.tight_layout()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()


def main():
    # Reproducibility
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    df = load_metadata(META_PATH)

    # Encode writer IDs -> integers
    le = LabelEncoder()
    df["writer_label"] = le.fit_transform(df["writer_id"].astype(str))
    class_names = le.classes_.tolist()
    num_classes = len(class_names)

    print("Total samples:", len(df))
    print("Num writers (classes):", num_classes)

    df_train, df_val, df_test = stratified_split(df, label_col="writer_id", seed=SEED)
    print("Split sizes:", len(df_train), len(df_val), len(df_test))

    # Labels for metrics need to align with the ds order
    # We'll build each ds directly from the respective df order.
    y_train = le.transform(df_train["writer_id"].astype(str))
    y_val = le.transform(df_val["writer_id"].astype(str))
    y_test = le.transform(df_test["writer_id"].astype(str))

    # Hyperparameter search (validation-based)
    grid = [
        {"lr": 1e-3, "batch_size": 64, "epochs": 10},
        {"lr": 3e-4, "batch_size": 64, "epochs": 12},
        {"lr": 1e-3, "batch_size": 128, "epochs": 10},
    ]

    best = None

    for i, hp in enumerate(grid, 1):
        print(f"\n=== Trial {i}/{len(grid)}: {hp} ===")

        train_ds = make_tf_dataset(
            df_train["image_path"].values, y_train,
            batch_size=hp["batch_size"], training=True
        )
        val_ds = make_tf_dataset(
            df_val["image_path"].values, y_val,
            batch_size=hp["batch_size"], training=False
        )

        model = build_2layer_cnn(num_classes=num_classes, lr=hp["lr"])

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=3, restore_best_weights=True
            )
        ]

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=hp["epochs"],
            callbacks=callbacks,
            verbose=1
        )

        val_acc, val_f1, _, _ = evaluate_model(model, val_ds, y_val)
        print(f"Validation accuracy: {val_acc:.4f}")
        print(f"Validation macro-F1: {val_f1:.4f}")

        score = val_f1  # choose best by macro-F1
        if best is None or score > best["score"]:
            best = {"score": score, "hp": hp}

    print("\nBest hyperparams by validation macro-F1:", best["hp"])
    print("Best validation macro-F1:", best["score"])

    # Retrain on combined train+val using best hp
    df_trainval = pd.concat([df_train, df_val], ignore_index=True)
    y_trainval = le.transform(df_trainval["writer_id"].astype(str))

    trainval_ds = make_tf_dataset(
        df_trainval["image_path"].values, y_trainval,
        batch_size=best["hp"]["batch_size"], training=True
    )
    test_ds = make_tf_dataset(
        df_test["image_path"].values, y_test,
        batch_size=best["hp"]["batch_size"], training=False
    )

    final_model = build_2layer_cnn(num_classes=num_classes, lr=best["hp"]["lr"])

    final_model.fit(
        trainval_ds,
        epochs=best["hp"]["epochs"],
        verbose=1
    )

    # Final evaluation on test EXACTLY ONCE
    test_acc, test_f1, cm, y_pred = evaluate_model(final_model, test_ds, y_test)

    print("\n=== FINAL TEST METRICS (run once) ===")
    print(f"Test accuracy: {test_acc:.4f}")
    print(f"Test macro-F1: {test_f1:.4f}")

    print("\nClassification report (macro emphasis):")
    print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

    plot_confusion_matrix(cm, class_names, max_classes=40)


if __name__ == "__main__":
    main()

Total samples: 8030
Num writers (classes): 45
Split sizes: 5620 1205 1205

=== Trial 1/3: {'lr': 0.001, 'batch_size': 64, 'epochs': 10} ===
Epoch 1/10
19/88 ━━━━━━━━━━━━━━━━━━━━ 14s 213ms/step - accuracy: 0.0286 - loss: 3.9898

KeyboardInterrupt: 